# CDNs, Plainly — the numbers behind Part 10

Companion notebook for **"System Design, Plainly — Part 10: CDNs"** (Nov 18, 2026).
Every cell runs standalone; the sims are toy models on this machine, the field
numbers are cited from Core Web Vitals' State of CWV (Nov 2026) and the CDN
market report (Oct 2026).

## 1. Distance is a latency floor

Light in fiber travels at ~200,000 km/s, so the round-trip floor is 10 µs per km.
No hardware, no protocol, no amount of money repeals it — the only move is to
put the bytes closer to the reader.

In [ ]:
C_FIBER_KM_S = 200_000  # km/s, speed of light in fiber

def rtt_floor_ms(distance_km):
    return 2 * distance_km / C_FIBER_KM_S * 1000  # round trip, ms

for d in (50, 500, 3000, 10000):
    print(f"{d:>6,} km -> RTT floor {rtt_floor_ms(d):6.1f} ms")

assert abs(rtt_floor_ms(50) - 0.5) < 1e-9
assert abs(rtt_floor_ms(3000) - 30.0) < 1e-9
assert abs(rtt_floor_ms(10000) - 100.0) < 1e-9
print("OK: edge at 50 km has a 60x lower floor than origin at 3,000 km")

## 2. What the edge is actually worth, measured

Chrome field data (Core Web Vitals, Nov 2026, 141,631 sites with phone field
data): median time to first byte is **691 ms** serving HTML direct from the
origin, **521 ms** via Cloudflare, **501 ms** via Fastly. Only 23.3% of sites
use an HTML edge CDN at all — and of those, 56.5% are Cloudflare, 14.6%
CloudFront, 14.1% Fastly, 11.2% Akamai.

In [ ]:
ttfb = {"no CDN": 691, "Cloudflare": 521, "Fastly": 501}
for name, v in ttfb.items():
    print(f"{name:>11}: {v} ms")
print(f"Cloudflare saves {(ttfb['no CDN'] - ttfb['Cloudflare']) / ttfb['no CDN'] * 100:.1f}% of TTFB")
print(f"Fastly saves     {(ttfb['no CDN'] - ttfb['Fastly']) / ttfb['no CDN'] * 100:.1f}% of TTFB")
assert abs((691 - 521) / 691 - 0.246) < 0.001
assert abs((691 - 501) / 691 - 0.275) < 0.001
# Good-LCP rate: 86% of Cloudflare-served sites vs 82% of all sites
print("Good LCP: 86% (Cloudflare) vs 82% (all sites) — a 4-point edge from TTFB alone")

## 3. Hit ratio is the only number that matters

Toy model: 1,000 requests/s, an edge hit costs 15 ms, an origin miss 200 ms.
Origin load = (1 − h) × 1,000; mean latency = h × 15 + (1 − h) × 200.
Going 90% → 99% cuts origin load 10× and halves mean latency; 99% → 99.9%
cuts origin load another 10× but barely moves latency — the tail is now the
misses you can't cache, not the ones you can.

In [ ]:
def origin_load_and_latency(hit_ratio, rps=1000, edge_ms=15, origin_ms=200):
    return (1 - hit_ratio) * rps, hit_ratio * edge_ms + (1 - hit_ratio) * origin_ms

for h in (0.90, 0.99, 0.999):
    load, lat = origin_load_and_latency(h)
    print(f"hit {h*100:5.1f}% -> origin {load:6.1f} rps, mean latency {lat:5.1f} ms")

load, lat = origin_load_and_latency(0.90)
assert abs(load - 100.0) < 1e-9 and abs(lat - 33.5) < 1e-9
load, lat = origin_load_and_latency(0.99)
assert abs(load - 10.0) < 1e-9 and abs(lat - 16.85) < 1e-9
load, lat = origin_load_and_latency(0.999)
assert abs(load - 1.0) < 1e-9 and abs(lat - 15.185) < 1e-9
print("OK: 90->99% is a 10x origin cut; 99->99.9% is another 10x, latency barely moves")

## 4. The shield: why one cache layer isn't enough

Flash crowd: 1,000,000 requests for 100 hot objects inside one TTL window.
No cache: the origin eats all 1M. Edge-only with 100 PoPs: each PoP fetches
each object once → 10,000 origin requests. Add a regional shield (10 regions):
each region fetches each object once → 1,000 origin requests. The shield is
a 1,000× cut on the origin, and it's the difference between a launch and an
outage.

In [ ]:
requests, hot_objects = 1_000_000, 100
no_cache = requests
edge_only = 100 * hot_objects        # 100 PoPs x 100 objects
shielded = 10 * hot_objects          # 10 regions x 100 objects
print(f"no cache : {no_cache:>7,} origin requests")
print(f"edge only: {edge_only:>7,} origin requests ({no_cache // edge_only}x cut)")
print(f"shielded : {shielded:>7,} origin requests ({no_cache // shielded}x cut)")
assert (no_cache, edge_only, shielded) == (1_000_000, 10_000, 1_000)
print("OK: shield turns a 1M-request flash crowd into 1,000 origin fetches")

## 5. Failure: one provider's bad day is your outage

On Nov 18, 2025 a malformed Cloudflare bot-management feature file returned
5xx errors worldwide for several hours. Availability arithmetic: a single
provider at 99.9% is 8.76 hours of downtime a year. Two independent 99.9%
providers with failover: 1 − (1 − 0.999)² = 99.9999%, about 32 seconds a year.
The catch is *independent* — if both CDNs sit behind the same DNS or the same
origin, the shared part is the real SLA.

And the cheap version of the same insurance: `stale-while-revalidate`. During
a 3-minute origin outage, a cache that serves stale for 24h keeps 100% of
cached-object requests succeeding; without it, every miss is a 5xx.

In [ ]:
MIN_PER_YEAR = 365.25 * 24 * 60
single = 0.001 * MIN_PER_YEAR          # 99.9%
dual = (0.001 ** 2) * MIN_PER_YEAR     # two independent 99.9% providers
print(f"single 99.9% CDN : {single:6.1f} min/yr  ({single/60:.2f} h)")
print(f"dual 99.9% + failover: {dual:6.2f} min/yr ({dual*60:.1f} s)")
assert abs(single - 525.96) < 0.01
assert abs(dual * 60 - 31.6) < 0.1

# Stale-while-revalidate during a 3-minute origin outage, 1,000 rps, 95% of
# requests are for objects already cached at the edge.
rps, outage_s, cached_share = 1000, 180, 0.95
served_stale = rps * outage_s * cached_share
failed_hard = rps * outage_s * (1 - cached_share)
print(f"outage: {served_stale:,.0f} requests served stale, {failed_hard:,.0f} fail")
assert abs(served_stale - 171_000) < 1e-6 and abs(failed_hard - 9_000) < 1e-6
print("OK: stale-while-revalidate turns a 180,000-request outage into 9,000 failures")

## Takeaways

- Distance sets a latency floor (10 µs/km RTT in fiber); the only fix is proximity.
- Measured payoff is real but modest on TTFB: ~25% (521 vs 691 ms median, Nov 2026 field data).
- Hit ratio is the lever: 90% → 99% cuts origin load 10×; 99% → 99.9% another 10×.
- A regional shield turns a 1M-request flash crowd into 1,000 origin fetches.
- Two independent CDNs: 8.76 h/yr → 32 s/yr of expected downtime. Shared DNS or origin voids the math.
- `stale-while-revalidate` is free outage insurance for cached objects.